<a href="https://colab.research.google.com/github/Prezzo-K/Graph-Neural-Networks-for-Fraud-Detection/blob/classical-ml-experiments/notebooks/Traditional_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import scikit learn and check version
import sklearn as sk

sk.__version__

'1.8.0'

In [2]:
# Workflow
# 1. Get the dataset, do preprocessing if any and split it into train and test set
# 2. Load various clasical ML models - Random Forest (RF), Extra Trees, Logistic Regression (LR), Support Vector Machine (SVM), XGBoost
# 3. Train and test them on the test set
# 3. Compare their Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix


# Load Dataset

In [3]:
import os
import pandas as pd

# Dynamically find the dataset path relative to the notebook's directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
file_path = os.path.join(project_root, "data", "Fraud Detection Transactions Dataset.csv")

dataset  = pd.read_csv(filepath_or_buffer = file_path)
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  str    
 1   User_ID                       50000 non-null  str    
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  str    
 4   Timestamp                     50000 non-null  str    
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  str    
 7   Location                      50000 non-null  str    
 8   Merchant_Category             50000 non-null  str    
 9   IP_Address_Flag               50000 non-null  int64  
 10  Previous_Fraudulent_Activity  50000 non-null  int64  
 11  Daily_Transaction_Count       50000 non-null  int64  
 12  Avg_Transaction_Amount_7d     50000 non-null  float64
 13  Failed_Trans

In [4]:
# Retrieve the feature and labels separate

X = dataset.loc[:,"Transaction_ID":"Is_Weekend"]
y = dataset.loc[:, "Fraud_Label"]

X.head(2)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend
0,TXN_33553,USER_1834,39.79,POS,2023-08-14 19:30:00,93213.17,Laptop,Sydney,Travel,0,0,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07 04:01:00,75725.25,Mobile,New York,Clothing,0,0,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0


In [5]:
y.head(2)

0    0
1    1
Name: Fraud_Label, dtype: int64

In [6]:
y.value_counts()

Fraud_Label
0    33933
1    16067
Name: count, dtype: int64

In [7]:
# Standardize columns with continous values - Transaction_Amount, Account_Balance, Daily_Transaction_Count, Avg_Transaction_Amount_7d, Card_Age, Transaction_Distance, Risk_Score
# Encode these as one hot encoding - Transaction_Type, Device_Type, Location, Merchant_Category, Card_Type, Authentication_Method
# Leave Binary / Numeric columns as they are. Don't scale or encode - IP_Address_Flag, Previous_Fraudulent_Activity, - Is_Weekend

# Drop the Transaction_ID, User_ID as they don't add in classification
X.drop(columns = ["Transaction_ID", "User_ID"], inplace = True)

# Remove risk score feature to avoid data leakge
# this feature seems to do data leakage though it will be beneficial in real world scenarios
# We also remove Failed_Transaction_Count_7d because it perfectly predicts fraud and dominates the models
X.drop(columns = ["Risk_Score", "Failed_Transaction_Count_7d"], inplace = True)

# Convert 'Timestamp' to datetime and extract various fields
X["Timestamp"] = pd.to_datetime(X["Timestamp"])
X["Hour"] = X["Timestamp"].dt.hour
X["Day"] = X["Timestamp"].dt.day
X["Month"] = X["Timestamp"].dt.month
X["DayOfWeek"] = X["Timestamp"].dt.dayofweek
# drop off timestamp col now
X.drop(columns = ["Timestamp"], inplace = True)

# Do one hot encoding
categorical_cols = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
    "Authentication_Method"
]
X = pd.get_dummies(X, columns = categorical_cols)
# Display the first few rows with the new 'Hour' column to confirm
X

# Scale the dataset
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


In [8]:
# split the data into train and test set
from sklearn.model_selection import train_test_split

random_state = 42
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = test_size,
                                                    shuffle = True,
                                                    stratify = y
                                                    )
len(X_train), len(y_train), len(X_test), len(y_test)

(40000, 40000, 10000, 10000)

In [9]:
# do some sanity check to ensure good splits of the classes in train and test sets
y_train.value_counts(), y_test.value_counts()

(Fraud_Label
 0    27146
 1    12854
 Name: count, dtype: int64,
 Fraud_Label
 0    6787
 1    3213
 Name: count, dtype: int64)

# Load Models And Train

## Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

# create the model with a fixed random state for reproducibility
rf = RandomForestClassifier(random_state=42)
# fit the model on the data
rf.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [11]:
preds_rf = rf.predict(X_test)

In [12]:
# metrics -  Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# List to hold results
model_results = []

def evaluate_model(y_true, preds, model_name="Model"):
    accuracy = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, zero_division=0)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    report = classification_report(y_true, preds, zero_division=0)

    print(f"\n{model_name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("\nClassification Report:\n", report)
    
    # Store the results
    result_dict = {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1-Score": f1,
        "Precision": precision,
        "Recall": recall
    }
    model_results.append(result_dict)
    
    return result_dict

# Random Forest Evaluation (which was created right before this cell typically)
evaluate_model(y_test, preds_rf, model_name="Random Forest Classifier")



Random Forest Classifier Metrics:
Accuracy: 0.6780
F1-Score: 0.0037
Precision: 0.3158
Recall: 0.0019

Classification Report:
               precision    recall  f1-score   support

           0       0.68      1.00      0.81      6787
           1       0.32      0.00      0.00      3213

    accuracy                           0.68     10000
   macro avg       0.50      0.50      0.41     10000
weighted avg       0.56      0.68      0.55     10000



{'Model': 'Random Forest Classifier',
 'Accuracy': 0.678,
 'F1-Score': 0.0037128712871287127,
 'Precision': 0.3157894736842105,
 'Recall': 0.0018674136321195146}

## Extra Trees

In [13]:
from sklearn.ensemble import ExtraTreesClassifier

# load extra tree classifier with a fixed random state
ext_clf = ExtraTreesClassifier(random_state=42)
# fit it in the data
ext_clf.fit(X_train, y_train)

# test on the test set
preds_ext_clf = ext_clf.predict(X_test)

# Evaluate the Extra Trees Classifier
evaluate_model(y_test, preds_ext_clf, model_name="Extra Trees Classifier")



Extra Trees Classifier Metrics:
Accuracy: 0.6738
F1-Score: 0.0320
Precision: 0.3439
Recall: 0.0168

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.98      0.80      6787
           1       0.34      0.02      0.03      3213

    accuracy                           0.67     10000
   macro avg       0.51      0.50      0.42     10000
weighted avg       0.57      0.67      0.56     10000



{'Model': 'Extra Trees Classifier',
 'Accuracy': 0.6738,
 'F1-Score': 0.032047477744807124,
 'Precision': 0.34394904458598724,
 'Recall': 0.01680672268907563}

## XGBoost

In [14]:
!pip install -q xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [15]:
from xgboost import XGBClassifier

# load XGBoost classifier with a fixed random state
xg_bst = XGBClassifier(random_state=42, eval_metric='logloss')
# fit it in the data
xg_bst.fit(X_train, y_train)

# test on the test set
xg_bst_preds = xg_bst.predict(X_test)

# Evaluate the XGBoost Classifier
evaluate_model(y_test, xg_bst_preds, model_name="XGBoost Classifier")



XGBoost Classifier Metrics:
Accuracy: 0.6608
F1-Score: 0.1069
Precision: 0.3470
Recall: 0.0632

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.94      0.79      6787
           1       0.35      0.06      0.11      3213

    accuracy                           0.66     10000
   macro avg       0.51      0.50      0.45     10000
weighted avg       0.57      0.66      0.57     10000



{'Model': 'XGBoost Classifier',
 'Accuracy': 0.6608,
 'F1-Score': 0.10689836756187467,
 'Precision': 0.347008547008547,
 'Recall': 0.06318082788671024}

## Logistic Regression

In [16]:
from sklearn.linear_model import LogisticRegression

# Load Logistic Regression with a fixed random state and longer iterations for scaled data convergence
lr = LogisticRegression(random_state=42, max_iter=1000)
# Fit it to the data
lr.fit(X_train, y_train)

# Test on the test set
preds_lr = lr.predict(X_test)

# Evaluate Logistic Regression
evaluate_model(y_test, preds_lr, model_name="Logistic Regression")



Logistic Regression Metrics:
Accuracy: 0.6787
F1-Score: 0.0000
Precision: 0.0000
Recall: 0.0000

Classification Report:
               precision    recall  f1-score   support

           0       0.68      1.00      0.81      6787
           1       0.00      0.00      0.00      3213

    accuracy                           0.68     10000
   macro avg       0.34      0.50      0.40     10000
weighted avg       0.46      0.68      0.55     10000



{'Model': 'Logistic Regression',
 'Accuracy': 0.6787,
 'F1-Score': 0.0,
 'Precision': 0.0,
 'Recall': 0.0}

## Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

# Load Support Vector Classifier
# Note: probability=True is needed if you want to calculate AUC-ROC later
svm_clf = SVC(random_state=42, probability=True)
# Fit it to the data
svm_clf.fit(X_train, y_train)

# Test on the test set
preds_svm = svm_clf.predict(X_test)

# Evaluate SVM
evaluate_model(y_test, preds_svm, model_name="SVM Classifier")


In [ ]:
import os

# Create the results directory if it doesn't exist
results_dir = os.path.join(os.path.dirname(os.getcwd()), "results")
os.makedirs(results_dir, exist_ok=True)

results_file_path = os.path.join(results_dir, "traditional_ml_results.txt")

# Write results to the file
with open(results_file_path, "w") as f:
    f.write("Traditional ML Models Performance\n")
    f.write("=================================\n\n")
    for res in model_results:
        f.write(f"Model: {res['Model']}\n")
        f.write(f"  Accuracy:  {res['Accuracy']:.4f}\n")
        f.write(f"  F1-Score:  {res['F1-Score']:.4f}\n")
        f.write(f"  Precision: {res['Precision']:.4f}\n")
        f.write(f"  Recall:    {res['Recall']:.4f}\n")
        f.write("-" * 35 + "\n")

print(f"Results successfully saved to {results_file_path}")
